# Samsung SAMSum Processed + FinDisputeEval EDA

This notebook is intended for VS Code with a Colab runtime. It mounts Google Drive, downloads `koushik7198/Samsung-samsum_processed` from Hugging Face, exports it to Drive, and runs EDA focused on FinDisputeEval summarization needs.

Source dataset: `koushik7198/Samsung-samsum_processed` on Hugging Face. The dataset viewer lists one default subset with three splits and two text columns: `input` and `output`.

## Why FinDisputeEval Should Look at Dialogue Summarization

FinDisputeEval is not using SAMSum as a financial-dispute corpus. SAMSum is useful because a dispute assistant must compress a multi-turn intake conversation into a concise, factual case summary for human handoff, audit logs, downstream case systems, and evaluation.

For FinDisputeEval, summarization training/evaluation helps answer:

- Can the model preserve who did what, when, and why across multiple turns?
- Does it keep critical slots such as amount, date, merchant, payment rail, card type, and disputed action?
- Does it avoid hallucinating missing facts or making unsupported policy conclusions?
- Can it distinguish facts, uncertainty, customer emotion, and requested next step?
- Can a generated handoff summary be short enough for operations while still complete enough for Reg E / Reg Z routing review?

Use SAMSum for summarization mechanics and evaluation scaffolding. Use CFPB + synthetic dispute dialogues for domain-positive financial-dispute content.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
%pip -q install -U datasets huggingface_hub pyarrow matplotlib rouge-score

In [ ]:
from __future__ import annotations

import json
import math
import os
import re
from collections import Counter
from pathlib import Path
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import DatasetDict, load_dataset, load_from_disk

DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_DIR = DRIVE_ROOT / "FinDisputeEval"
RUN_ID = globals().get("RUN_ID", datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
OUTPUT_DIR = PROJECT_DIR / "dataset" / "interim" / "samsum"
RAW_EXPORT_DIR = PROJECT_DIR / "dataset" / "external" / "samsum" / "raw" / "hf_dataset"
PROCESSED_DIR = OUTPUT_DIR / "processed_findispute"
EDA_DIR = PROJECT_DIR / "outputs" / "data_pipeline" / "samsum_summarization_eda" / "eda_v01" / f"run_{RUN_ID}_colab"
CACHE_DIR = PROJECT_DIR / "dataset" / "cache" / "huggingface"

for directory in [OUTPUT_DIR, RAW_EXPORT_DIR, PROCESSED_DIR, EDA_DIR, CACHE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(CACHE_DIR)
os.environ["HF_DATASETS_CACHE"] = str(CACHE_DIR / "datasets")

DATASET_ID = "koushik7198/Samsung-samsum_processed"
RANDOM_STATE = 42

print(f"Dataset: {DATASET_ID}")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
ds = load_dataset(DATASET_ID)
ds.save_to_disk(str(RAW_EXPORT_DIR))

manifest = {
    "dataset_id": DATASET_ID,
    "output_dir": str(OUTPUT_DIR),
    "hf_dataset_dir": str(RAW_EXPORT_DIR),
    "splits": {},
}

print(ds)
for split_name, split_ds in ds.items():
    manifest["splits"][split_name] = {
        "rows": len(split_ds),
        "features": list(split_ds.features.keys()),
        "jsonl": str(OUTPUT_DIR / f"{split_name}.jsonl"),
        "csv": str(OUTPUT_DIR / f"{split_name}.csv"),
        "parquet": str(OUTPUT_DIR / f"{split_name}.parquet"),
    }
    split_ds.to_json(str(OUTPUT_DIR / f"{split_name}.jsonl"), orient="records", lines=True, force_ascii=False)
    split_ds.to_csv(str(OUTPUT_DIR / f"{split_name}.csv"))
    split_ds.to_parquet(str(OUTPUT_DIR / f"{split_name}.parquet"))

(OUTPUT_DIR / "download_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(f"Manifest saved to: {OUTPUT_DIR / 'download_manifest.json'}")
display(pd.DataFrame([
    {"split": split_name, **split_info}
    for split_name, split_info in manifest["splits"].items()
]))

## Normalize Input / Output Pairs

In [ ]:
def safe_text(value) -> str:
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    return str(value)


def normalize_space(text: str) -> str:
    return re.sub(r"\s+", " ", safe_text(text)).strip()


def word_count(text: str) -> int:
    text = normalize_space(text)
    return len(text.split()) if text else 0


def sentence_count(text: str) -> int:
    text = normalize_space(text)
    if not text:
        return 0
    parts = re.split(r"(?<=[.!?])\s+", text)
    return len([part for part in parts if part.strip()])


def parse_dialogue_turns(text: str) -> list[dict]:
    """Parse SAMSum-style `Speaker: utterance` text into coarse turns.

    This parser is intentionally conservative. It supports repeated speakers and
    multiline continuation text without trying to infer missing speakers.
    """
    text = safe_text(text).replace("\r\n", "\n").replace("\r", "\n")
    turns = []
    current_speaker = None
    current_parts = []
    speaker_pattern = re.compile(r"^\s*([^:\n]{1,60}):\s*(.*)$")

    for raw_line in text.split("\n"):
        line = raw_line.strip()
        if not line:
            continue
        match = speaker_pattern.match(line)
        if match:
            if current_speaker is not None:
                turns.append({"speaker": current_speaker, "utterance": normalize_space(" ".join(current_parts))})
            current_speaker = match.group(1).strip()
            current_parts = [match.group(2).strip()]
        else:
            current_parts.append(line)

    if current_speaker is not None:
        turns.append({"speaker": current_speaker, "utterance": normalize_space(" ".join(current_parts))})
    return turns


def speaker_stats(text: str) -> dict:
    turns = parse_dialogue_turns(text)
    speakers = [turn["speaker"] for turn in turns if turn.get("speaker")]
    return {
        "turn_count": len(turns),
        "speaker_count": len(set(speakers)),
        "speakers": " | ".join(sorted(set(speakers))),
        "avg_turn_words": float(np.mean([word_count(turn["utterance"]) for turn in turns])) if turns else 0.0,
        "max_turn_words": max([word_count(turn["utterance"]) for turn in turns], default=0),
    }


frames = []
for split_name, split_ds in ds.items():
    df = split_ds.to_pandas()
    if not {"input", "output"}.issubset(df.columns):
        raise ValueError(f"Expected columns input/output, got {df.columns.tolist()}")
    df["split"] = split_name
    frames.append(df)

samsum_df = pd.concat(frames, ignore_index=True)
samsum_df["row_id"] = [f"{row.split}-{i}" for i, row in enumerate(samsum_df.itertuples(index=False))]
samsum_df["dialogue"] = samsum_df["input"].map(safe_text)
samsum_df["summary"] = samsum_df["output"].map(safe_text)
samsum_df["dialogue_norm"] = samsum_df["dialogue"].map(normalize_space)
samsum_df["summary_norm"] = samsum_df["summary"].map(normalize_space)
samsum_df["dialogue_words"] = samsum_df["dialogue_norm"].map(word_count)
samsum_df["summary_words"] = samsum_df["summary_norm"].map(word_count)
samsum_df["dialogue_chars"] = samsum_df["dialogue_norm"].str.len()
samsum_df["summary_chars"] = samsum_df["summary_norm"].str.len()
samsum_df["summary_sentences"] = samsum_df["summary_norm"].map(sentence_count)
samsum_df["compression_ratio_words"] = samsum_df["summary_words"] / samsum_df["dialogue_words"].replace(0, np.nan)

speaker_feature_df = pd.DataFrame([speaker_stats(text) for text in samsum_df["dialogue"]])
samsum_df = pd.concat([samsum_df, speaker_feature_df], axis=1)

samsum_df.to_parquet(PROCESSED_DIR / "samsum_processed_normalized.parquet", index=False)
samsum_df.to_csv(PROCESSED_DIR / "samsum_processed_normalized.csv", index=False)

print(f"Rows: {len(samsum_df):,}")
display(samsum_df[["row_id", "split", "dialogue_words", "summary_words", "compression_ratio_words", "turn_count", "speaker_count", "dialogue", "summary"]].head(10))

## Dataset Shape + Summarization EDA

In [ ]:
split_summary = (
    samsum_df.groupby("split", dropna=False)
    .agg(
        rows=("row_id", "count"),
        avg_dialogue_words=("dialogue_words", "mean"),
        p50_dialogue_words=("dialogue_words", "median"),
        p95_dialogue_words=("dialogue_words", lambda s: s.quantile(0.95)),
        avg_summary_words=("summary_words", "mean"),
        p50_summary_words=("summary_words", "median"),
        p95_summary_words=("summary_words", lambda s: s.quantile(0.95)),
        avg_compression_ratio=("compression_ratio_words", "mean"),
        avg_turn_count=("turn_count", "mean"),
        avg_speaker_count=("speaker_count", "mean"),
    )
    .reset_index()
)

turn_summary = (
    samsum_df.groupby("split", dropna=False)
    .agg(
        p50_turns=("turn_count", "median"),
        p95_turns=("turn_count", lambda s: s.quantile(0.95)),
        max_turns=("turn_count", "max"),
        p50_speakers=("speaker_count", "median"),
        p95_speakers=("speaker_count", lambda s: s.quantile(0.95)),
        max_speakers=("speaker_count", "max"),
    )
    .reset_index()
)

length_bucket_bins = [0, 50, 100, 150, 250, 500, 1000, 100000]
length_bucket_labels = ["<=50", "51-100", "101-150", "151-250", "251-500", "501-1000", ">1000"]
samsum_df["dialogue_length_bucket"] = pd.cut(samsum_df["dialogue_words"], bins=length_bucket_bins, labels=length_bucket_labels, include_lowest=True)
length_bucket_summary = (
    samsum_df.groupby(["split", "dialogue_length_bucket"], observed=True)
    .size()
    .reset_index(name="rows")
)

split_summary.to_csv(EDA_DIR / "samsum_split_summary.csv", index=False)
turn_summary.to_csv(EDA_DIR / "samsum_turn_summary.csv", index=False)
length_bucket_summary.to_csv(EDA_DIR / "samsum_length_bucket_summary.csv", index=False)

display(split_summary)
display(turn_summary)
display(length_bucket_summary)

## FinDisputeEval-Oriented Feature Scan

These features look for dialogue-summary phenomena relevant to dispute handoff quality. Keyword hits are not labels.

In [ ]:
KEYWORD_GROUPS = {
    "money_payment": [r"\$\s?\d+", r"\bmoney\b", r"\bpaid\b", r"\bpay\b", r"payment", r"refund", r"charge", r"bill", r"cost", r"price"],
    "account_access": [r"account", r"password", r"login", r"log in", r"email", r"phone", r"address"],
    "dispute_like": [r"dispute", r"complain", r"complaint", r"fraud", r"stolen", r"lost", r"wrong", r"unauthori[sz]ed", r"scam"],
    "transaction_timing": [r"today", r"yesterday", r"tomorrow", r"monday", r"tuesday", r"wednesday", r"thursday", r"friday", r"saturday", r"sunday", r"\b\d{1,2}:\d{2}\b"],
    "commitment_next_step": [r"will", r"going to", r"need to", r"has to", r"should", r"must", r"agreed", r"decided"],
    "uncertainty_correction": [r"maybe", r"think", r"not sure", r"actually", r"instead", r"rather", r"sorry", r"mistake"],
    "emotion_escalation": [r"angry", r"upset", r"mad", r"annoyed", r"frustrated", r"fuck", r"shit", r"ridiculous"],
    "attachment_media": [r"<file", r"photo", r"video", r"gif", r"document", r"scan"],
}

for group_name, patterns in KEYWORD_GROUPS.items():
    regex = "|".join(patterns)
    samsum_df[f"kw_{group_name}"] = samsum_df["dialogue_norm"].str.lower().str.contains(regex, regex=True, na=False)

keyword_cols = [f"kw_{group_name}" for group_name in KEYWORD_GROUPS]
samsum_df["keyword_hit_count"] = samsum_df[keyword_cols].sum(axis=1)

def assign_findispute_summary_role(row) -> str:
    if row["kw_money_payment"] and row["kw_dispute_like"]:
        return "near_dispute_summary_probe"
    if row["kw_money_payment"]:
        return "money_or_payment_summary_style"
    if row["kw_account_access"]:
        return "account_access_summary_style"
    if row["kw_commitment_next_step"]:
        return "next_step_commitment_summary_style"
    if row["kw_uncertainty_correction"]:
        return "uncertainty_or_correction_summary_style"
    if row["turn_count"] >= 15 or row["speaker_count"] >= 4:
        return "long_multi_party_compression"
    return "general_dialogue_summary_style"


samsum_df["findispute_summary_role"] = samsum_df.apply(assign_findispute_summary_role, axis=1)

keyword_summary = (
    samsum_df.melt(
        id_vars=["split", "findispute_summary_role"],
        value_vars=keyword_cols,
        var_name="keyword_group",
        value_name="hit",
    )
    .query("hit")
    .groupby(["keyword_group", "split"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values(["keyword_group", "rows"], ascending=[True, False])
)

role_summary = (
    samsum_df.groupby(["findispute_summary_role"], dropna=False)
    .agg(
        rows=("row_id", "count"),
        avg_dialogue_words=("dialogue_words", "mean"),
        avg_summary_words=("summary_words", "mean"),
        avg_compression_ratio=("compression_ratio_words", "mean"),
        avg_turns=("turn_count", "mean"),
        avg_speakers=("speaker_count", "mean"),
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

role_by_split = (
    samsum_df.groupby(["split", "findispute_summary_role"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values(["split", "rows"], ascending=[True, False])
)

samsum_df.to_parquet(PROCESSED_DIR / "samsum_processed_with_findispute_features.parquet", index=False)
samsum_df.to_csv(PROCESSED_DIR / "samsum_processed_with_findispute_features.csv", index=False)
keyword_summary.to_csv(EDA_DIR / "samsum_findispute_keyword_summary.csv", index=False)
role_summary.to_csv(EDA_DIR / "samsum_findispute_role_summary.csv", index=False)
role_by_split.to_csv(EDA_DIR / "samsum_findispute_role_by_split.csv", index=False)

display(role_summary)
display(role_by_split)
display(keyword_summary.head(50))

## Summary Quality Proxy Checks

These checks flag examples that are especially useful for FinDispute handoff-summary evaluation: very compressed summaries, long dialogues, multi-party dialogues, and examples where the summary may omit numbers or high-salience facts.

In [ ]:
NUMBER_PATTERN = re.compile(r"(?:\$\s?\d+(?:[,.]\d+)?|\b\d+(?:[,.]\d+)?\b)")

def extract_numbers(text: str) -> set[str]:
    return {match.group(0).lower().replace(" ", "") for match in NUMBER_PATTERN.finditer(safe_text(text))}


dialogue_numbers = samsum_df["dialogue"].map(extract_numbers)
summary_numbers = samsum_df["summary"].map(extract_numbers)
samsum_df["dialogue_number_count"] = dialogue_numbers.map(len)
samsum_df["summary_number_count"] = summary_numbers.map(len)
samsum_df["numbers_missing_from_summary"] = [len(d - s) for d, s in zip(dialogue_numbers, summary_numbers)]
samsum_df["has_number_omission_risk"] = samsum_df["numbers_missing_from_summary"] > 0
samsum_df["very_high_compression"] = samsum_df["compression_ratio_words"] < 0.12
samsum_df["long_dialogue"] = samsum_df["dialogue_words"] >= samsum_df["dialogue_words"].quantile(0.90)
samsum_df["multi_party"] = samsum_df["speaker_count"] >= 4

quality_proxy_summary = pd.DataFrame([
    {"proxy_flag": "has_number_omission_risk", "rows": int(samsum_df["has_number_omission_risk"].sum()), "rate": float(samsum_df["has_number_omission_risk"].mean())},
    {"proxy_flag": "very_high_compression", "rows": int(samsum_df["very_high_compression"].sum()), "rate": float(samsum_df["very_high_compression"].mean())},
    {"proxy_flag": "long_dialogue", "rows": int(samsum_df["long_dialogue"].sum()), "rate": float(samsum_df["long_dialogue"].mean())},
    {"proxy_flag": "multi_party", "rows": int(samsum_df["multi_party"].sum()), "rate": float(samsum_df["multi_party"].mean())},
])

risk_examples = samsum_df[
    samsum_df[["has_number_omission_risk", "very_high_compression", "long_dialogue", "multi_party"]].any(axis=1)
].copy()

quality_proxy_summary.to_csv(EDA_DIR / "samsum_quality_proxy_summary.csv", index=False)
risk_examples[[
    "row_id", "split", "findispute_summary_role", "dialogue_words", "summary_words",
    "compression_ratio_words", "turn_count", "speaker_count", "numbers_missing_from_summary",
    "dialogue", "summary",
]].to_csv(EDA_DIR / "samsum_quality_proxy_examples.csv", index=False)

display(quality_proxy_summary)
display(risk_examples[[
    "row_id", "split", "findispute_summary_role", "dialogue_words", "summary_words",
    "compression_ratio_words", "turn_count", "speaker_count", "numbers_missing_from_summary",
    "dialogue", "summary",
]].head(30))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(17, 11))

split_summary.set_index("split")["rows"].plot(kind="bar", ax=axes[0, 0], title="Rows by split")
axes[0, 0].set_xlabel("")
axes[0, 0].set_ylabel("rows")

samsum_df["dialogue_words"].clip(upper=samsum_df["dialogue_words"].quantile(0.99)).plot(
    kind="hist", bins=40, ax=axes[0, 1], title="Dialogue length distribution (words, clipped p99)"
)
axes[0, 1].set_xlabel("dialogue words")

samsum_df["compression_ratio_words"].clip(upper=samsum_df["compression_ratio_words"].quantile(0.99)).plot(
    kind="hist", bins=40, ax=axes[1, 0], title="Summary compression ratio (clipped p99)"
)
axes[1, 0].set_xlabel("summary words / dialogue words")

role_summary.sort_values("rows").plot(
    kind="barh", x="findispute_summary_role", y="rows", ax=axes[1, 1], legend=False, title="FinDispute summary-role coverage"
)
axes[1, 1].set_xlabel("rows")
axes[1, 1].set_ylabel("")

plt.tight_layout()
overview_path = EDA_DIR / "samsum_findispute_eda_overview.png"
fig.savefig(overview_path, dpi=160, bbox_inches="tight")
print(f"Saved plot to: {overview_path}")
plt.show()

## FinDispute Summary Seed Samples

These samples are for learning and evaluating dialogue-summary mechanics. They are not financial-dispute labels.

In [ ]:
sample_specs = [
    ("near_dispute_summary_probe", 30),
    ("money_or_payment_summary_style", 30),
    ("account_access_summary_style", 25),
    ("next_step_commitment_summary_style", 25),
    ("uncertainty_or_correction_summary_style", 25),
    ("long_multi_party_compression", 25),
    ("general_dialogue_summary_style", 20),
]

seed_parts = []
used = set()
for role_name, target_n in sample_specs:
    pool = samsum_df[
        samsum_df["findispute_summary_role"].eq(role_name)
        & ~samsum_df["row_id"].isin(used)
    ]
    if pool.empty:
        print(f"No rows for role: {role_name}")
        continue
    n = min(target_n, len(pool))
    sampled = pool.sample(n=n, random_state=RANDOM_STATE).copy()
    sampled["seed_bucket"] = role_name
    used.update(sampled["row_id"].tolist())
    seed_parts.append(sampled)

if seed_parts:
    seed_sample = pd.concat(seed_parts, ignore_index=True).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
else:
    seed_sample = samsum_df.sample(n=min(100, len(samsum_df)), random_state=RANDOM_STATE).copy()
    seed_sample["seed_bucket"] = "fallback_random"

seed_cols = [
    "seed_bucket", "row_id", "split", "findispute_summary_role", "dialogue_words", "summary_words",
    "compression_ratio_words", "turn_count", "speaker_count", "keyword_hit_count",
    "numbers_missing_from_summary", "dialogue", "summary",
]

seed_path = EDA_DIR / "samsum_findispute_summary_seed_sample.csv"
seed_sample[seed_cols].to_csv(seed_path, index=False)

print(f"Saved seed sample: {seed_path}")
print(f"Seed rows: {len(seed_sample):,}")
display(seed_sample[seed_cols].head(30))

## Expected Output Structure

```text
dataset/external/samsum/raw/hf_dataset/

dataset/interim/samsum/
  train.jsonl / train.csv / train.parquet
  validation.jsonl / validation.csv / validation.parquet
  test.jsonl / test.csv / test.parquet
  download_manifest.json
  processed_findispute/
    samsum_processed_normalized.csv / .parquet
    samsum_processed_with_findispute_features.csv / .parquet

outputs/data_pipeline/samsum_summarization_eda/eda_v01/<run_id>/
  samsum_split_summary.csv
  samsum_turn_summary.csv
  samsum_length_bucket_summary.csv
  samsum_findispute_keyword_summary.csv
  samsum_findispute_role_summary.csv
  samsum_quality_proxy_summary.csv
  samsum_quality_proxy_examples.csv
  samsum_findispute_eda_overview.png
  samsum_findispute_summary_seed_sample.csv
```
